# 多模型实体识别评测与领域分析

计算和展示精确率(P)、召回率(R)、F1值，并按领域进行分析

In [1]:
import json
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
from tqdm.notebook import tqdm
import hashlib
from typing import Dict, List, Tuple
from collections import defaultdict
from IPython.display import display, HTML

# 设置图表风格
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False

## 数据加载与处理

In [2]:
def load_data(file_path: str) -> List[Dict]:
    """加载JSON数据"""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    
    # 尝试直接解析
    try:
        data = json.loads(content)
        return data if isinstance(data, list) else [data]
    except json.JSONDecodeError:
        pass
    
    # 尝试从markdown代码块中提取
    json_match = re.search(r'```(?:json)?\s*([^`]*?)\s*```', content)
    if json_match:
        json_content = json_match.group(1).strip()
        try:
            data = json.loads(json_content)
            return data if isinstance(data, list) else [data]
        except json.JSONDecodeError:
            pass
    
    # 尝试修复不完整的JSON数组
    if '[' in content and '"ne"' in content:
        try:
            start_idx = content.find('[')
            if start_idx != -1:
                json_part = content[start_idx:]
                lines = json_part.split('\n')
                valid_lines = ['[']
                
                for line in lines[1:]:
                    line = line.strip()
                    if not line or line == '[':
                        continue
                    if line.startswith('"') and (line.endswith('",') or line.endswith('"')):
                        valid_lines.append('  ' + line)
                    elif line == ']':
                        break
                
                if len(valid_lines) > 1 and valid_lines[-1].endswith(','):
                    valid_lines[-1] = valid_lines[-1][:-1]
                
                valid_lines.append(']')
                fixed_json = '\n'.join(valid_lines)
                data = json.loads(fixed_json)
                return data if isinstance(data, list) else [data]
        except:
            pass
    
    raise ValueError(f"无法解析文件: {file_path}")

def create_unique_id(item: Dict) -> str:
    """创建唯一ID"""
    if 'text' in item:
        text = item['text'].strip()
    elif 'data' in item and 'text' in item['data']:
        text = item['data']['text'].strip()
    else:
        text = str(item)
    
    return hashlib.md5(text.encode()).hexdigest()[:8]

In [3]:
def clean_entity_text(text: str) -> str:
    """清理实体文本"""
    if not text:
        return ""
    
    text = str(text).strip()
    # 去除换行符和制表符
    text = re.sub(r'[\n\r\t]+', ' ', text)
    # 去除多余的空格
    text = re.sub(r'\s+', ' ', text)
    # 去除首尾的引号
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1].strip()
    # 去除首尾的方括号
    if text.startswith('[') and text.endswith(']'):
        text = text[1:-1].strip()
    
    return text

def is_valid_entity(entity: str) -> bool:
    """检查实体是否有效"""
    if not entity:
        return False
    
    # 跳过无效词汇
    invalid_words = {'ne', 'null', 'none', 'na', 'n/a', 'unknown', 'undefined', 
                    'empty', 'blank', 'void', '无', '空', '未知', 'nil'}
    if entity.lower() in invalid_words:
        return False
    
    # 跳过过短或过长的实体
    if len(entity) < 1 or len(entity) > 200:
        return False
    
    # 跳过只包含标点符号
    if all(c in '.,;:!?-_()[]{}' for c in entity):
        return False
    
    # 跳过包含控制字符
    if any(ord(c) < 32 for c in entity):
        return False
    
    return True

In [4]:
def extract_entities_from_label_studio(data: List[Dict]) -> Dict[str, List[str]]:
    """从Label Studio格式提取实体"""
    entities = {}
    
    for item in data:
        unique_id = create_unique_id(item)
        entity_list = []
        
        annotations = item.get('annotations', [])
        for annotation in annotations:
            results = annotation.get('result', [])
            for result in results:
                value = result.get('value', {})
                text = value.get('text')
                if text:
                    cleaned_text = clean_entity_text(text)
                    if is_valid_entity(cleaned_text):
                        entity_list.append(cleaned_text)
        
        entities[unique_id] = list(dict.fromkeys(entity_list))  # 去重保持顺序
    
    return entities

def extract_entities(data: List[Dict], key: str) -> Dict[str, List[str]]:
    """提取预测实体"""
    entities = {}
    
    for item in data:
        unique_id = create_unique_id(item)
        entity_list = []
        entities_data = item.get(key, [])
        
        if isinstance(entities_data, str):
            # 处理markdown代码块
            markdown_match = re.search(r'```(?:json)?\s*([^`]*?)\s*```', entities_data)
            if markdown_match:
                json_content = markdown_match.group(1).strip()
                try:
                    parsed_entities = json.loads(json_content)
                    if isinstance(parsed_entities, list):
                        entity_list.extend(parsed_entities)
                    else:
                        entity_list.append(str(parsed_entities))
                except json.JSONDecodeError:
                    if json_content:
                        entity_list.append(json_content)
            else:
                try:
                    # 处理不完整的JSON数组
                    cleaned_data = entities_data.strip()
                    
                    if cleaned_data.startswith('[') and not cleaned_data.rstrip().endswith(']'):
                        lines = cleaned_data.split('\n')
                        valid_lines = []
                        for line in lines:
                            line = line.strip()
                            if not line or line == '[':
                                continue
                            if (line.startswith('"') and (line.endswith('",') or line.endswith('"'))) or line == ']':
                                valid_lines.append(line)
                            else:
                                break
                        
                        if valid_lines:
                            json_content = '[\n  ' + '\n  '.join(valid_lines) + '\n]'
                            json_content = re.sub(r',(\s*\])', r'\1', json_content)
                            try:
                                parsed_entities = json.loads(json_content)
                                if isinstance(parsed_entities, list):
                                    entity_list.extend(parsed_entities)
                            except json.JSONDecodeError:
                                pass
                    else:
                        parsed_entities = json.loads(cleaned_data)
                        if isinstance(parsed_entities, list):
                            entity_list.extend(parsed_entities)
                        else:
                            entity_list.append(str(parsed_entities))
                except json.JSONDecodeError:
                    entity_list.append(entities_data)
        elif isinstance(entities_data, list):
            for ent in entities_data:
                if isinstance(ent, dict) and 'text' in ent:
                    entity_list.append(ent['text'])
                else:
                    entity_list.append(str(ent))
        
        # 清理和过滤实体
        cleaned_entities = []
        for entity in entity_list:
            cleaned_entity = clean_entity_text(str(entity))
            if is_valid_entity(cleaned_entity):
                cleaned_entities.append(cleaned_entity)
        
        entities[unique_id] = list(dict.fromkeys(cleaned_entities))  # 去重保持顺序
    
    return entities

## 指标计算

In [5]:
def compute_metrics(true_entities: Dict[str, List[str]], pred_entities: Dict[str, List[str]]) -> Tuple[float, float, float]:
    """计算P、R、F1"""
    tp = 0
    fp = 0
    fn = 0
    
    for id_val in true_entities:
        true_set = set(true_entities.get(id_val, []))
        pred_set = set(pred_entities.get(id_val, []))

        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)
    
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    
    return precision, recall, f1

def compute_domain_metrics(true_entities: Dict[str, List[str]], pred_entities: Dict[str, List[str]], 
                          domains: Dict[str, str]) -> Dict[str, Dict]:
    """计算各领域的评测指标"""
    domain_tp = defaultdict(int)
    domain_fp = defaultdict(int)
    domain_fn = defaultdict(int)
    domain_samples = defaultdict(list)
    
    for id_val in true_entities:
        true_set = set(true_entities.get(id_val, []))
        pred_set = set(pred_entities.get(id_val, []))
        
        domain = domains.get(id_val, "未知")
        domain_samples[domain].append(id_val)

        tp_current = len(true_set & pred_set)
        fp_current = len(pred_set - true_set)
        fn_current = len(true_set - pred_set)
        
        domain_tp[domain] += tp_current
        domain_fp[domain] += fp_current
        domain_fn[domain] += fn_current
    
    domain_metrics = {}
    for domain in domain_tp:
        tp = domain_tp[domain]
        fp = domain_fp[domain]
        fn = domain_fn[domain]
        
        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        
        domain_metrics[domain] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": tp + fn,
            "sample_count": len(domain_samples[domain])
        }
    
    return domain_metrics

## 模型评测

In [6]:
def evaluate_model(gold_file: str, pred_file: str, gold_key: str = 'label', pred_key: str = 'predicted_entities') -> Dict:
    """评估单个模型"""
    # 加载数据
    gold_data = load_data(gold_file)
    pred_data = load_data(pred_file)

    # 检查是否为Label Studio格式
    is_label_studio_format = any('annotations' in item for item in gold_data)
    
    # 提取实体
    if is_label_studio_format:
        gold_entities = extract_entities_from_label_studio(gold_data)
    else:
        gold_entities = extract_entities(gold_data, gold_key)
    
    pred_entities = extract_entities(pred_data, pred_key)
    
    # 提取领域信息
    domains = {}
    for item in gold_data:
        unique_id = create_unique_id(item)
        domains[unique_id] = item.get('domain', "未知")
    
    # 计算指标
    precision, recall, f1 = compute_metrics(gold_entities, pred_entities)
    domain_metrics = compute_domain_metrics(gold_entities, pred_entities, domains)
    
    model_name = os.path.basename(pred_file).replace(".json", "")
    
    return {
        "model_name": model_name,
        "overall": {
            "precision": precision,
            "recall": recall,
            "f1": f1
        },
        "domain_metrics": domain_metrics
    }

def evaluate_all_models(gold_file: str, pred_folder: str, gold_key: str = 'label', pred_key: str = 'predicted_entities') -> Dict[str, Dict]:
    """评估所有模型"""
    pred_files = glob.glob(os.path.join(pred_folder, "*.json"))
    
    # 排除金标准文件
    gold_basename = os.path.basename(gold_file)
    pred_files = [f for f in pred_files if os.path.basename(f) != gold_basename]
    
    if not pred_files:
        print(f"在 {pred_folder} 中没有找到预测文件")
        return {}
    
    all_metrics = {}
    for pred_file in tqdm(pred_files, desc="评估模型"):
        try:
            metrics = evaluate_model(gold_file, pred_file, gold_key, pred_key)
            all_metrics[metrics["model_name"]] = metrics
            print(f"模型 {metrics['model_name']} 评测完成, F1: {metrics['overall']['f1']:.4f}")
        except Exception as e:
            print(f"评估模型 {os.path.basename(pred_file)} 时出错: {str(e)}")
    
    return all_metrics

## 结果展示

In [7]:
def create_overall_metrics_table(all_metrics: Dict[str, Dict]) -> pd.DataFrame:
    """创建总体指标对比表格"""
    data = []
    for model_name, metrics in all_metrics.items():
        overall = metrics["overall"]
        data.append({
            "模型": model_name,
            "精确率(P)": f"{overall['precision']:.4f}",
            "召回率(R)": f"{overall['recall']:.4f}",
            "F1值": f"{overall['f1']:.4f}"
        })
    
    df = pd.DataFrame(data)
    if not df.empty:
        df['F1_numeric'] = df['F1值'].astype(float)
        df = df.sort_values('F1_numeric', ascending=False).drop('F1_numeric', axis=1).reset_index(drop=True)
    
    return df

def create_domain_comparison_table(all_metrics: Dict[str, Dict]) -> pd.DataFrame:
    """创建领域对比表格"""
    # 获取所有领域
    all_domains = set()
    for metrics in all_metrics.values():
        if "domain_metrics" in metrics:
            all_domains.update(metrics["domain_metrics"].keys())
    
    all_domains = sorted(list(all_domains))
    if "未知" in all_domains:
        all_domains.remove("未知")
        all_domains.append("未知")
    
    # 创建表格数据
    table_data = []
    for model_name, metrics in all_metrics.items():
        model_data = {"模型": model_name}
        
        # 添加各领域F1值
        for domain in all_domains:
            if ("domain_metrics" in metrics and domain in metrics["domain_metrics"]):
                f1 = metrics["domain_metrics"][domain]["f1"]
                model_data[f"{domain}"] = f"{f1:.4f}"
            else:
                model_data[f"{domain}"] = "N/A"
        
        # 添加总体F1值
        overall_f1 = metrics["overall"]["f1"]
        model_data["总体"] = f"{overall_f1:.4f}"
        
        table_data.append(model_data)
   
    df = pd.DataFrame(table_data)
    if not df.empty:
        df = df.sort_values(by="总体", ascending=False).reset_index(drop=True)
    
    return df

def create_domain_detail_tables(all_metrics: Dict[str, Dict]) -> Dict[str, pd.DataFrame]:
    """创建每个领域的详细表格"""
    all_domains = set()
    for metrics in all_metrics.values():
        if "domain_metrics" in metrics:
            all_domains.update(metrics["domain_metrics"].keys())
    
    domain_tables = {}
    
    for domain in all_domains:
        domain_data = []
        for model_name, metrics in all_metrics.items():
            if "domain_metrics" in metrics and domain in metrics["domain_metrics"]:
                domain_metrics = metrics["domain_metrics"][domain]
                domain_data.append({
                    "模型": model_name,
                    "精确率(P)": f"{domain_metrics['precision']:.4f}",
                    "召回率(R)": f"{domain_metrics['recall']:.4f}",
                    "F1值": f"{domain_metrics['f1']:.4f}",
                    "样本数": domain_metrics['sample_count'],
                    "支持度": domain_metrics['support']
                })
        
        if domain_data:
            df = pd.DataFrame(domain_data)
            df['F1_numeric'] = df['F1值'].astype(float)
            df = df.sort_values('F1_numeric', ascending=False).drop('F1_numeric', axis=1).reset_index(drop=True)
            domain_tables[domain] = df
    
    return domain_tables

In [8]:
def visualize_overall_performance(all_metrics: Dict[str, Dict]):
    """可视化总体性能对比"""
    if not all_metrics:
        print("没有模型数据可供可视化")
        return
    
    models = []
    precision_values = []
    recall_values = []
    f1_values = []
    
    for model_name, metrics in all_metrics.items():
        overall = metrics["overall"]
        models.append(model_name)
        precision_values.append(overall["precision"])
        recall_values.append(overall["recall"])
        f1_values.append(overall["f1"])
    
    # 按F1值排序
    sorted_indices = np.argsort(f1_values)[::-1]
    models = [models[i] for i in sorted_indices]
    precision_values = [precision_values[i] for i in sorted_indices]
    recall_values = [recall_values[i] for i in sorted_indices]
    f1_values = [f1_values[i] for i in sorted_indices]
    
    plt.figure(figsize=(12, 8))
    x = np.arange(len(models))
    width = 0.25
    
    plt.bar(x - width, precision_values, width, label='精确率', color='#3498db', alpha=0.8)
    plt.bar(x, recall_values, width, label='召回率', color='#2ecc71', alpha=0.8)
    plt.bar(x + width, f1_values, width, label='F1值', color='#9b59b6', alpha=0.8)
    
    plt.xlabel('模型')
    plt.ylabel('得分')
    plt.title('模型总体性能对比')
    plt.xticks(x, models, rotation=45, ha='right')
    plt.ylim(0, 1.0)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def visualize_domain_performance(all_metrics: Dict[str, Dict]):
    """可视化领域性能对比"""
    all_domains = set()
    all_models = list(all_metrics.keys())
    
    for metrics in all_metrics.values():
        if "domain_metrics" in metrics:
            all_domains.update(metrics["domain_metrics"].keys())
    
    if not all_domains:
        print("没有领域数据可供可视化")
        return
    
    all_domains = sorted(list(all_domains))
    
    # 提取每个模型在各领域的F1值
    domain_f1_values = {}
    for model_name in all_models:
        domain_f1_values[model_name] = []
        for domain in all_domains:
            if ("domain_metrics" in all_metrics[model_name] and 
                domain in all_metrics[model_name]["domain_metrics"]):
                f1 = all_metrics[model_name]["domain_metrics"][domain]["f1"]
            else:
                f1 = 0
            domain_f1_values[model_name].append(f1)
    
    plt.figure(figsize=(14, 8))
    x = np.arange(len(all_domains))
    width = 0.8 / len(all_models)
    
    for i, model_name in enumerate(all_models):
        offset = (i - len(all_models) / 2 + 0.5) * width
        plt.bar(x + offset, domain_f1_values[model_name], width, 
                label=model_name, alpha=0.7)
    
    plt.xlabel('领域')
    plt.ylabel('F1值')
    plt.title('各模型在不同领域的F1值对比')
    plt.xticks(x, all_domains, rotation=45, ha='right')
    plt.ylim(0, 1.0)
    plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

## 主程序

In [9]:
def main(gold_file, pred_folder, gold_key='label', pred_key='predicted_entities'):
    """主程序"""
    # 评估所有模型
    all_metrics = evaluate_all_models(gold_file, pred_folder, gold_key, pred_key)
    
    if not all_metrics:
        print("没有找到有效的预测结果文件。")
        return
    
    # 1. 总体性能对比
    display(HTML("<h2>模型总体性能对比</h2>"))
    overall_df = create_overall_metrics_table(all_metrics)
    display(overall_df.style.set_properties(**{'text-align': 'center'}).set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'center')]}]
    ))
    
    # 2. 可视化总体性能
    visualize_overall_performance(all_metrics)
    
    # 3. 领域性能汇总对比
    domain_comparison_df = create_domain_comparison_table(all_metrics)
    if not domain_comparison_df.empty and len(domain_comparison_df.columns) > 2:  # 除了模型列和总体列还有其他领域列
        display(HTML("<h2>领域性能汇总对比（F1值）</h2>"))
        display(domain_comparison_df.style.set_properties(**{'text-align': 'center'}).set_table_styles(
            [{'selector': 'th', 'props': [('text-align', 'center')]}]
        ))
        
        # 4. 可视化领域性能
        visualize_domain_performance(all_metrics)
        
        # 5. 各领域详细性能
        domain_detail_tables = create_domain_detail_tables(all_metrics)
        if domain_detail_tables:
            display(HTML("<h2>各领域详细性能对比</h2>"))
            for domain, df in domain_detail_tables.items():
                display(HTML(f"<h3>领域: {domain}</h3>"))
                display(df.style.set_properties(**{'text-align': 'center'}).set_table_styles(
                    [{'selector': 'th', 'props': [('text-align', 'center')]}]
                ))
    
    return all_metrics

In [ ]:
# 运行评估 - 请修改为您的文件路径



#gold_file = "data/Test-Manua-Chem_0818-50.json" 

gold_file = "data/Bio-test-data-0226-100.json"
pred_folder = "data/test_0801"  # 预测结果文件夹路径


results = main(gold_file, pred_folder)

if results:
    print(f"\n评估完成！共评估了 {len(results)} 个模型。")
else:
    print("没有找到有效的预测结果文件。")

评估模型:   0%|          | 0/4 [00:00<?, ?it/s]

评估模型 0225-Abstract_BIO_predictions-7B_100-origin.json 时出错: [Errno 2] No such file or directory: 'Bio-test-data-0226-100.json'
评估模型 0225-Abstract_BIO_predictions-7B_100_trained_processed.json 时出错: [Errno 2] No such file or directory: 'Bio-test-data-0226-100.json'
评估模型 0225-deepseek_100.json 时出错: [Errno 2] No such file or directory: 'Bio-test-data-0226-100.json'
评估模型 0225-gpt-5_2-chat-latest_100_results.json 时出错: [Errno 2] No such file or directory: 'Bio-test-data-0226-100.json'
没有找到有效的预测结果文件。
没有找到有效的预测结果文件。
